# Notebook 07 — Feature Leakage and Scoring-Time Audit

**Purpose:** review the nine features used by the saved Random Forest and document whether each should be available before a transaction is approved.

This is a practical audit, not an automatic proof that leakage is absent. PaySim is simulated data, so document assumptions and confirm them against the intended real-world scoring workflow.


## 1. Load the feature dataset and saved model metadata

Run this notebook from the project root or the `notebooks` folder. It reads the existing processed feature data and the metadata created in Notebook 06.


In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists() and (PROJECT_ROOT.parent / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "transactions_features.csv.gz"
METADATA_PATH = PROJECT_ROOT / "models" / "fraud_detection_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Model metadata not found: {METADATA_PATH}")

df = pd.read_csv(DATA_PATH, compression="gzip")
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

FEATURES = metadata["feature_columns"]
TARGET = metadata["target_column"]
TIME_COL = metadata["time_column"]

print("Dataset shape:", df.shape)
print("Target:", TARGET)
print("Time column:", TIME_COL)
print("Features from saved model:", len(FEATURES))
print(FEATURES)


Dataset shape: (6362620, 22)
Target: isFraud
Time column: step
Features from saved model: 9
['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour_of_day', 'night_flag', 'amount_to_origin_balance_ratio', 'origin_balance_was_zero', 'origin_txn_count_prior_steps']


## 2. Confirm that model features exist and contain usable values

In [2]:

missing_features = [c for c in FEATURES if c not in df.columns]
assert not missing_features, f"Features missing from dataset: {missing_features}"
assert TARGET in df.columns, f"Target column missing: {TARGET}"

audit_rows = []
for col in FEATURES:
    s = df[col]
    numeric_s = pd.to_numeric(s, errors="coerce")
    is_numeric = pd.api.types.is_numeric_dtype(s)
    audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_count": int(s.isna().sum()),
        "missing_pct": round(float(s.isna().mean() * 100), 6),
        "unique_values": int(s.nunique(dropna=True)),
        "numeric_infinite_count": int(np.isinf(numeric_s.dropna()).sum()) if is_numeric else None,
        "sample_values": str(s.dropna().head(3).tolist())
    })

feature_quality = pd.DataFrame(audit_rows)
display(feature_quality)

quality_path = OUTPUT_DIR / "feature_quality_audit.csv"
feature_quality.to_csv(quality_path, index=False)
print("Saved:", quality_path)


,feature,dtype,missing_count,missing_pct,unique_values,numeric_infinite_count,sample_values
0,type,str,0,0.0,5,NaN,"['PAYMENT', 'PAYMENT', 'TRANSFER']"
1,amount,float64,0,0.0,5316900,0.0,"[9839.64, 1864.28, 181.0]"
2,oldbalanceOrg,float64,0,0.0,1845844,0.0,"[170136.0, 21249.0, 181.0]"
3,oldbalanceDest,float64,0,0.0,3614697,0.0,"[0.0, 0.0, 0.0]"
4,hour_of_day,int64,0,0.0,24,0.0,"[1, 1, 1]"
5,night_flag,int64,0,0.0,2,0.0,"[1, 1, 1]"
6,amount_to_origin_balance_ratio,float64,0,0.0,6208443,0.0,"[0.0578339681196219, 0.0877349522330462, 1.0]"
7,origin_balance_was_zero,int64,0,0.0,2,0.0,"[0, 0, 0]"
8,origin_txn_count_prior_steps,int64,0,0.0,3,0.0,"[0, 0, 0]"


Saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\feature_quality_audit.csv


## 3. Review feature availability at scoring time

The table below is a **human-review checklist**. It does not infer operational availability from data alone. 


In [3]:

review_guidance = {
    "type": {
        "expected_source": "Transaction request",
        "initial_assessment": "Likely available before decision",
        "question": "Is transaction type present in the authorization request?"
    },
    "amount": {
        "expected_source": "Transaction request",
        "initial_assessment": "Likely available before decision",
        "question": "Is requested amount available before authorization?"
    },
    "oldbalanceOrg": {
        "expected_source": "Origin account state before transaction",
        "initial_assessment": "Confirm operational timing",
        "question": "Is the balance snapshot captured before this transaction is applied?"
    },
    "oldbalanceDest": {
        "expected_source": "Destination account state before transaction",
        "initial_assessment": "Confirm operational timing",
        "question": "Is destination balance known and available before this transaction is applied?"
    },
    "hour_of_day": {
        "expected_source": "Derived from simulation step",
        "initial_assessment": "Simulation-specific",
        "question": "Can the equivalent time feature be computed at decision time in the real system?"
    },
    "night_flag": {
        "expected_source": "Derived from hour_of_day",
        "initial_assessment": "Simulation-specific",
        "question": "Is the time zone and night-hour definition specified for deployment?"
    },
    "amount_to_origin_balance_ratio": {
        "expected_source": "Amount divided by old origin balance",
        "initial_assessment": "Confirm zero handling and timing",
        "question": "Is it calculated only from requested amount and the pre-transaction origin balance?"
    },
    "origin_balance_was_zero": {
        "expected_source": "Derived from old origin balance",
        "initial_assessment": "Likely pre-transaction if based on oldbalanceOrg",
        "question": "Does this feature use oldbalanceOrg only, not newbalanceOrig?"
    },
    "origin_txn_count_prior_steps": {
        "expected_source": "Historical transactions before current step",
        "initial_assessment": "Audit grouping and ordering",
        "question": "Does the count exclude the current transaction and all transactions at the same step?"
    }
}

review_rows = []
for feature in FEATURES:
    info = review_guidance.get(feature, {
        "expected_source": "Not in standard review list",
        "initial_assessment": "Manual review required",
        "question": "Document exact calculation and whether it is available before scoring."
    })
    review_rows.append({
        "feature": feature,
        **info,
        "review_status": "PENDING — confirm with project design"
    })

availability_review = pd.DataFrame(review_rows)
display(availability_review)

review_path = OUTPUT_DIR / "feature_scoring_time_review.csv"
availability_review.to_csv(review_path, index=False)
print("Saved review checklist:", review_path)


,feature,expected_source,initial_assessment,question,review_status
0,type,Transaction request,Likely available before decision,Is transaction type present in the authorizati...,PENDING — confirm with project design
1,amount,Transaction request,Likely available before decision,Is requested amount available before authoriza...,PENDING — confirm with project design
2,oldbalanceOrg,Origin account state before transaction,Confirm operational timing,Is the balance snapshot captured before this t...,PENDING — confirm with project design
3,oldbalanceDest,Destination account state before transaction,Confirm operational timing,Is destination balance known and available bef...,PENDING — confirm with project design
4,hour_of_day,Derived from simulation step,Simulation-specific,Can the equivalent time feature be computed at...,PENDING — confirm with project design
5,night_flag,Derived from hour_of_day,Simulation-specific,Is the time zone and night-hour definition spe...,PENDING — confirm with project design
6,amount_to_origin_balance_ratio,Amount divided by old origin balance,Confirm zero handling and timing,Is it calculated only from requested amount an...,PENDING — confirm with project design
7,origin_balance_was_zero,Derived from old origin balance,Likely pre-transaction if based on oldbalanceOrg,"Does this feature use oldbalanceOrg only, not ...",PENDING — confirm with project design
8,origin_txn_count_prior_steps,Historical transactions before current step,Audit grouping and ordering,Does the count exclude the current transaction...,PENDING — confirm with project design


Saved review checklist: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\feature_scoring_time_review.csv


## 4. Check whether suspicious target or post-transaction columns entered the model




In [4]:

forbidden_features = [
    TARGET,
    "isFlaggedFraud",
    "nameOrig",
    "nameDest",
    "step",
    "newbalanceOrig",
    "newbalanceDest",
    "origin_balance_error",
    "destination_balance_error",
    "origin_balance_error_abs",
    "destination_balance_error_abs",
    "origin_emptied",
    "destination_balance_zero",
]

forbidden_found = [c for c in FEATURES if c in forbidden_features]

print("Forbidden or high-risk columns found in model features:", forbidden_found)
assert len(forbidden_found) == 0, (
    "Review model features: forbidden or high-risk columns are present."
)
print("PASS: No columns from the explicit forbidden list are included.")


Forbidden or high-risk columns found in model features: []
PASS: No columns from the explicit forbidden list are included.


## 5. Inspect fraud rates and distributions by feature
These summaries help investigate suspiciously predictive features. Strong separation is not proof of leakage, but it should be understood.

In [5]:

fraud_rate = float(df[TARGET].mean())
print(f"Overall fraud prevalence: {fraud_rate:.6%}")

numeric_features = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
categorical_features = [c for c in FEATURES if c not in numeric_features]

numeric_summary = []
for col in numeric_features:
    normal_values = pd.to_numeric(df.loc[df[TARGET] == 0, col], errors="coerce")
    fraud_values = pd.to_numeric(df.loc[df[TARGET] == 1, col], errors="coerce")
    numeric_summary.append({
        "feature": col,
        "normal_median": normal_values.median(),
        "fraud_median": fraud_values.median(),
        "normal_mean": normal_values.mean(),
        "fraud_mean": fraud_values.mean(),
        "normal_missing_pct": normal_values.isna().mean() * 100,
        "fraud_missing_pct": fraud_values.isna().mean() * 100
    })

numeric_summary_df = pd.DataFrame(numeric_summary)
display(numeric_summary_df)
numeric_summary_path = OUTPUT_DIR / "feature_distribution_by_target.csv"
numeric_summary_df.to_csv(numeric_summary_path, index=False)
print("Saved:", numeric_summary_path)

for col in categorical_features:
    print(f"\nFraud rate by {col}:")
    display(
        df.groupby(col, dropna=False)[TARGET]
          .agg(transactions="size", fraud_count="sum", fraud_rate="mean")
          .sort_values("fraud_rate", ascending=False)
          .head(20)
    )


Overall fraud prevalence: 0.129082%


,feature,normal_median,fraud_median,normal_mean,fraud_mean,normal_missing_pct,fraud_missing_pct
0,amount,74684.720000,441423.44,1.781970e+05,1.467967e+06,0.0,0.0
1,oldbalanceOrg,14069.000000,438983.45,8.328287e+05,1.649668e+06,0.0,0.0
2,oldbalanceDest,133311.800000,0.00,1.101421e+06,5.442496e+05,0.0,0.0
3,hour_of_day,16.000000,12.00,1.532633e+01,1.154645e+01,0.0,0.0
4,night_flag,0.000000,0.00,7.014769e-02,3.250944e-01,0.0,0.0
5,amount_to_origin_balance_ratio,6.512585,1.00,7.077315e+04,1.161967e+03,0.0,0.0
6,origin_balance_was_zero,0.000000,0.00,3.308583e-01,4.992086e-03,0.0,0.0
7,origin_txn_count_prior_steps,0.000000,0.00,1.457728e-03,2.191647e-03,0.0,0.0


Saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\feature_distribution_by_target.csv

Fraud rate by type:


,transactions,fraud_count,fraud_rate
type,,,
TRANSFER,532909,4097,0.007688
CASH_OUT,2237500,4116,0.001840
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


## 6. Check correlations with the target (numeric features only)
Correlation is only a screening tool. A high correlation does not automatically mean leakage, and a low correlation does not guarantee safety.

In [6]:

corr_data = df[FEATURES + [TARGET]].copy()
for col in corr_data.columns:
    if not pd.api.types.is_numeric_dtype(corr_data[col]):
        corr_data[col] = corr_data[col].astype("category").cat.codes

correlations = corr_data.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(
    key=lambda s: s.abs(), ascending=False
)
display(correlations.to_frame("correlation_with_isFraud"))

corr_path = OUTPUT_DIR / "feature_target_correlations.csv"
correlations.rename("correlation_with_isFraud").to_csv(corr_path)
print("Saved:", corr_path)


,correlation_with_isFraud
amount,0.076688
night_flag,0.035764
hour_of_day,-0.031403
origin_balance_was_zero,-0.024874
type,0.020833
oldbalanceOrg,0.010154
oldbalanceDest,-0.005885
amount_to_origin_balance_ratio,-0.004916
origin_txn_count_prior_steps,0.000689


Saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\feature_target_correlations.csv


## 7. Time-aware split sanity check

In [7]:

split_config_path = OUTPUT_DIR / "split_config.json"
if split_config_path.exists():
    with open(split_config_path, "r", encoding="utf-8") as f:
        split_config = json.load(f)
    cutoff = int(split_config["cutoff_step"])
    train_part = df.loc[df[TIME_COL] <= cutoff]
    test_part = df.loc[df[TIME_COL] > cutoff]
    print("Outer cutoff step:", cutoff)
    print("Train maximum step:", train_part[TIME_COL].max())
    print("Test minimum step:", test_part[TIME_COL].min())
    print("Train rows:", len(train_part))
    print("Test rows:", len(test_part))
    print("Chronological separation valid:", train_part[TIME_COL].max() < test_part[TIME_COL].min())
else:
    print("Split config not found; skipped split sanity check.")


Outer cutoff step: 594
Train maximum step: 594
Test minimum step: 595
Train rows: 6239040
Test rows: 123580
Chronological separation valid: True


## 8. Audit decision and sign-off




In [8]:

print("NOTEBOOK 07 — FEATURE AUDIT SUMMARY")
print("-" * 45)
print("Saved model feature count:", len(FEATURES))
print("Missing feature names:", missing_features)
print("Forbidden/high-risk features found:", forbidden_found)
print("Feature quality report saved:", quality_path.exists())
print("Scoring-time review checklist saved:", review_path.exists())
print("Target distribution report saved:", numeric_summary_path.exists())
print("Correlation report saved:", corr_path.exists())

print("\nStatus: AUTOMATED CHECKS COMPLETED; HUMAN FEATURE REVIEW STILL REQUIRED.")


NOTEBOOK 07 — FEATURE AUDIT SUMMARY
---------------------------------------------
Saved model feature count: 9
Missing feature names: []
Forbidden/high-risk features found: []
Feature quality report saved: True
Scoring-time review checklist saved: True
Target distribution report saved: True
Correlation report saved: True

Status: AUTOMATED CHECKS COMPLETED; HUMAN FEATURE REVIEW STILL REQUIRED.
